# Evidencia ejecutada de la Actividad 5 — auditoría sin reentrenamiento

**Objeto de esta bitácora:** verificar con código real y salidas conservadas los artefactos producidos en **Windows 11 / Python 3.13.5 / MLflow 3.17.1**. Este cuaderno se ejecuta en **GitHub Actions (Python 3.13)** de forma independiente y en modo **solo lectura**: no ejecuta GridSearchCV, no vuelve a entrenar, no crea nuevos runs de MLflow y no altera los resultados del experimento original.

Para entrenar desde cero, ver [`fuentes/entrena.ipynb`](entrena.ipynb), [`fuentes/train.py`](train.py) y la guía del [README](../README.md).

## 1. Identificación del entorno y carga de evidencias
Se verifica la existencia de los CSV, manifiestos y resultados exportados, y se documenta el entorno del revisor.

In [ ]:
import json, platform, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
CWD = Path.cwd().resolve()
ROOT = next((d for d in (CWD, *CWD.parents) if (d/"resultados/comparacion_modelos.csv").exists()), None)
assert ROOT is not None, "No se encontró raíz del repositorio"
R = ROOT/"resultados"
quality = json.loads((R/"calidad_datos.json").read_text(encoding="utf-8"))
experiment = json.loads((R/"seleccion_congelada.json").read_text(encoding="utf-8"))
manifest = json.loads((ROOT/"evidencias/mlflow/runs_manifest.json").read_text(encoding="utf-8"))
print("Notebook de auditoría ejecutado en:", platform.platform(), "| Python", sys.version.split()[0])
print("Experimento de origen:", experiment["metadata"]["platform"], "| Python", experiment["metadata"]["python_version"])
print("Dataset:", quality["filas"], "registros; 339 fallas; semilla",experiment["metadata"]["seed"])
print("Modelo congelado ANTES de test:",experiment["modelo_seleccionado"],"umbral:",experiment["umbral_congelado"])

## 2. Versionamiento, calidad y correspondencia con UCI
El conjunto tiene 10,000 filas y 339 fallas. El snapshot académico posee siete columnas; se registran SHA256 y ausencia de duplicados o nulos. Si el CSV UCI original ya fue incorporado, se valida también su existencia y correspondencia con el manifiesto.

In [ ]:
import hashlib
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):h.update(block)
    return h.hexdigest()
assert quality["filas"]==10000
assert quality["clases"]["1"]==339
assert quality["duplicados_exactos"]==0
assert sum(quality["nulos_por_variable"].values())==0
# Los hashes SHA256 en los resultados corresponden a la copia Windows original;
# al clonar, Git puede normalizar CRLF a LF. Comparar el CONTENIDO de CSV,
# no exigir igualdad byte a byte entre sistemas operativos.
original=pd.read_csv(ROOT/"datos/datos_ini/ai4i2020.csv")
prepared=pd.read_csv(ROOT/"datos/datos_limp/ai4i2020_limpio.csv")
splits=pd.read_csv(ROOT/"datos/splits.csv")
assert original.shape==(10000,7) and prepared.shape==(10000,8)
assert prepared["indice_original"].tolist()==list(range(10000))
assert original.reset_index(drop=True).equals(prepared.drop(columns=["indice_original"]).reset_index(drop=True))
assert set(splits["indice_original"])==set(range(10000))
assert (splits["machine_failure"].to_numpy()==original.loc[splits["indice_original"],"Machine failure"].to_numpy()).all()
print("Nota reproducibilidad: SHA256 de Windows pueden variar tras checkout LF/CRLF.")
print("SHA256 original Windows registrado:",quality["sha256_origen"])
print("SHA256 original en este checkout:",sha256(ROOT/"datos/datos_ini/ai4i2020.csv"))
full=ROOT/"docs/uci_source_manifest.json"
if full.exists():
    uci=json.loads(full.read_text(encoding="utf-8"))
    assert uci["comparacion_fila_a_fila"]=="APROBADA"
    assert sha256(ROOT/uci["archivo_original_uci"])==uci["sha256_original_uci"]
    print("Archivo original UCI:",uci["original_filas"],"filas /",uci["original_columnas"],"columnas")
else:
    print("Manifiesto UCI completo no disponible en este commit; verificar por separado.")
print("Hashes, integridad de datos y 0 faltantes: OK")
display(pd.DataFrame([{"variable":k,"outliers_IQR":v} for k,v in quality["iqr_outliers_por_variable"].items()]))

## 3. Validación estadística de predicciones y matrices
Se reconstruyen TP, TN, FP y FN de **las 1,500 predicciones reales** guardadas, y se comparan con las métricas oficiales sin estimaciones inventadas.

In [ ]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,average_precision_score
a=pd.read_csv(R/"comparacion_modelos.csv")
p=pd.read_csv(R/"predicciones_test.csv")
assert len(a)==2 and len(p)==1500
rows=[]
for _,r in a.iterrows():
    m=r["Modelo"]
    y=p["valor_real"].astype(int).to_numpy()
    pred=p[f"{m}_prediccion"].astype(int).to_numpy()
    probs=p[f"{m}_probabilidad"].astype(float).to_numpy()
    tp=int(((y==1)&(pred==1)).sum())
    tn=int(((y==0)&(pred==0)).sum())
    fp=int(((y==0)&(pred==1)).sum())
    fn=int(((y==1)&(pred==0)).sum())
    assert (tp,tn,fp,fn)==tuple(int(r["test_"+x]) for x in ("tp","tn","fp","fn"))
    assert np.isclose(f1_score(y,pred),r["test_f1"])
    assert np.isclose(recall_score(y,pred),r["test_recall"])
    assert np.isclose(precision_score(y,pred),r["test_precision"])
    assert np.isclose(average_precision_score(y,probs),r["test_average_precision"])
    assert np.array_equal(pred,(probs>=r["validation_threshold"]).astype(int))
    rows.append({"Modelo":m,"TP":tp,"FN":fn,"FP":fp,"TN":tn,
                 "Recall":round(r["test_recall"],4),"F1":round(r["test_f1"],4),
                 "AP":round(r["test_average_precision"],4)})
display(pd.DataFrame(rows))
print("Todas las matrices y métricas coinciden; test separado: OK")

## 4. Hipótesis comparativa y bootstrap pareado
La ventaja observada de Random Forest es descriptiva sobre el dataset sintético. La diferencia F1 tiene un intervalo bootstrap publicado, cuya consistencia se verifica.

In [ ]:
boot=pd.read_csv(R/"bootstrap_pareado.csv").iloc[0]
rf=a.set_index("Modelo").loc["RandomForest"]
lr=a.set_index("Modelo").loc["LogisticRegression"]
delta=rf["test_f1"]-lr["test_f1"]
assert np.isclose(boot["delta_observada"],delta)
assert int(boot["repeticiones"])==2000
assert float(boot["IC95_inferior"])<float(boot["IC95_superior"])
display(pd.DataFrame([{"Comparación":"RF - regresión logística",
                       "Delta F1":round(delta,4),
                       "IC95 inferior":round(boot["IC95_inferior"],4),
                       "IC95 superior":round(boot["IC95_superior"],4),
                       "Remuestreos":int(boot["repeticiones"])}]))
print("Interpretación: incertidumbre estimada en test; no es validación industrial.")

## 5. Trazabilidad de los 17 runs MLflow
Se contrastan las ejecuciones padre/hijo exportadas desde el MLflow de Windows; este notebook no interactúa con el servidor localhost.

In [ ]:
runs=pd.read_csv(ROOT/"evidencias/mlflow/runs_exportados.csv")
assert len(runs)==17==int(manifest["numero_runs"])
assert int(manifest["runs_hijos"])==14
parents=runs["parent_run_id"].fillna("").astype(str)
assert int((parents!="").sum())==14
assert runs["run_id"].nunique()==17
assert set(experiment["runs_mlflow"].values()).issubset(set(runs["run_id"]))
assert set(runs["estado"])=={"FINISHED"}
display(runs[["nombre","estado","parent_run_id"]].fillna("").head(17))
print("MLflow: 17 IDs únicos, 14 candidatos hijos, 3 runs base; todos finalizados.")

## 6. Conclusión y control de alcance
La auditoría confirma integridad de datos, métricas, particiones y metadatos MLflow **sin reentrenar**. La ejecución del entrenamiento está demostrada por el script original, los runs/artefactos y las capturas auténticas; el notebook de entrenamiento `entrena.ipynb` se conserva como plantilla reproducible. Ninguna salida de este cuaderno pretende atribuir a GitHub Actions el entrenamiento efectuado en Windows.

In [ ]:
assert experiment["modelo_seleccionado"]=="RandomForest"
assert np.isclose(experiment["umbral_congelado"],0.2)
assert set((ROOT/"evidencias/mlflow").glob("*.png")) >= {
    ROOT/"evidencias/mlflow/01_experimentos.png",
    ROOT/"evidencias/mlflow/02_gridsearch_parametros.png",
    ROOT/"evidencias/mlflow/03_comparacion_runs.png",
    ROOT/"evidencias/mlflow/04_modelos_artefactos.png",
}
print("AUDITORÍA REAL EJECUTADA Y COMPLETADA. Fuente de resultados: CSV/JSON publicados.")
print("No se recalibró ningún modelo ni se alteraron los resultados de la Actividad 5.")